In [2]:
import glob #importing glob library
import matplotlib.pyplot as plt #Need for plotting at the end
import numpy as np #numpi

In [3]:
import astropy
import astropy.units as u
from astropy.utils.data import download_file
from astropy.io import fits
from astropy.utils import data
from astropy.wcs import wcs #everything above is for reading FIT/FITS
from astropy.visualization import (AsinhStretch, LogStretch, 
                                  PercentileInterval, ImageNormalize, simple_norm, ZScaleInterval) #tools for visualizing large range of fits data

In [4]:
from astropy.nddata import CCDData
import pandas as pd
from astropy.coordinates import SkyCoord #don't remeber if any of this is needed

import ipywidgets as widgets
!pip install ipycanvas
from IPython.display import display #widgets/canvas is how the image gets displayed/interacted with

In [5]:
!pip install astropy astroalign numpy
import astroalign as aa #JJ said we would make our own huzzah

In [6]:
from ipycanvas import Canvas
from PIL import Image
from PIL import ImageTk
from tkinter import * #more canvas functions we will need

In [7]:
import astropy.units as u
from astropy.utils import data
!pip install spectral-cube
from spectral_cube import SpectralCube #turned out to be not needed but might need in the future

In [8]:
!pip install photutils
from photutils.aperture import CircularAperture
from photutils.aperture import aperture_photometry

In [118]:
import requests
import json
import time
import os

In [137]:
import requests
import json
L = requests.post('http://nova.astrometry.net/api/login', data={'request-json': json.dumps({"apikey": "dsfmjmbczhpwrrno"})})
login = L.json()
print(login)
session_key = login["session"]

{'status': 'success', 'message': 'authenticated user: isaacpohl@college.harvard.edu', 'session': '1w9xbbho1rn4fp0f9nhhvdn9vhmp3dpy'}


In [119]:
image_files = glob.glob("D:/EB NSVS01031772 Usable Files/2022 EB Files/03-22-2019/pipelineout/*.fit")

sub_ids = []

json_string = json.dumps({"session": f'{session_key}'})

for i in image_files:

    image_upload = {"request-json": (None, json_string, 'text/plain'), "file": (os.path.basename(i), open(i, 'rb'), 'application/octet-stream')}

    P = requests.post('http://nova.astrometry.net/api/upload/', files=image_upload)
    sub = P.json()
    print(sub)
    sub_id = sub['subid']
    sub_ids.append(sub_id)
print(sub_ids)

{'status': 'success', 'subid': 15635288, 'hash': '8174dc2c85dbd6c9c14f8fa00919284e3bf69257'}
{'status': 'success', 'subid': 15635289, 'hash': 'df2edffb74de18f58ff1d43939ae1fc49116cd33'}
{'status': 'success', 'subid': 15635290, 'hash': '069cfe5919f976d1e3c3dd2e70f1106139c45355'}
{'status': 'success', 'subid': 15635292, 'hash': '37d538a09dce35946fe1704cc9f25f70898be880'}
{'status': 'success', 'subid': 15635298, 'hash': '83d53c0bec341b5643f239fbc0c3a1dffcf17f03'}
{'status': 'success', 'subid': 15635299, 'hash': '8249fdfdbae3406859c7b3324c8c7ebb577ab772'}
{'status': 'success', 'subid': 15635300, 'hash': 'eef4378cafab5eb1a6c115ebe770b155bd671ae1'}
{'status': 'success', 'subid': 15635301, 'hash': '772876a2a28b596dd2c3f73d3893a0c03487bdb7'}
{'status': 'success', 'subid': 15635303, 'hash': '4f74542e73dafd9a6f193c0d8e8ecc4914583d2b'}
{'status': 'success', 'subid': 15635304, 'hash': 'f75d4dedf991196eabccdeb0bc9d020271c06700'}
{'status': 'success', 'subid': 15635305, 'hash': '77dc9cec1d241b7afeb5

In [130]:
job_ids = []

for i in range(len(sub_ids)):
    move_forward = False
    while not move_forward:
        S = requests.get(f'http://nova.astrometry.net/api/submissions/{sub_ids[i]}')
        get_id = S.json()
        if not get_id['jobs'] or None in get_id['jobs'] or if get_id['jobs'] is None: #this is because you have to check if the jobs array has not been created yet, if it is empty, or if it is filled with the string "None"
            time.sleep(5)
        else:
            move_forward = True
    job_ids.append(get_id['jobs'][0])

print(job_ids)

[[16450993], [16450994], [16451037], [16451049], [16451001], [16451067], [16451050], [16451016], [16451079], [16451054], [16451021], [16451006], [16451031], [16451059], [16451081]]


In [133]:
wcs_per_file = {f'wcs_{p+1}': {} for p in range(len(job_ids))}

for i in range(len(job_ids)):
    G = requests.get(f'http://nova.astrometry.net/api/jobs/{job_ids[i]}/calibration/')
    wcs_info = G.json()
    wcs_dict = wcs_per_file[f'wcs_{i+1}']
    if 'ra' in wcs_info:
        wcs_dict['ra'] = wcs_info['ra']
    else:
        print(f'wcs_{i+1} missing ra')
    if 'dec' in wcs_info:
        wcs_dict['dec'] = wcs_info['dec']
    else:
        print(f'wcs_{i+1} missing dec')
    if 'pixscale' in wcs_info:
        wcs_dict['arcsecs_pix'] = wcs_info['pixscale']
    else:
        print(f'wcs_{i+1} missing pixscale')
wcs_per_file    

{'wcs_1': {'ra': 206.4241079445839,
  'dec': 79.3663646735679,
  'arcsecs_pix': 0.8201355075772486},
 'wcs_2': {'ra': 206.42537579399442,
  'dec': 79.36611606931007,
  'arcsecs_pix': 0.8201730728024932},
 'wcs_3': {'ra': 206.42504544177606,
  'dec': 79.36618027445387,
  'arcsecs_pix': 0.820167711577099},
 'wcs_4': {'ra': 206.4227017556861,
  'dec': 79.36607385857997,
  'arcsecs_pix': 0.8202298533186253},
 'wcs_5': {'ra': 206.41958727236428,
  'dec': 79.36607768884902,
  'arcsecs_pix': 0.8202477380554389},
 'wcs_6': {'ra': 206.41987271355652,
  'dec': 79.36596961006465,
  'arcsecs_pix': 0.8202309240012386},
 'wcs_7': {'ra': 206.41885674807222,
  'dec': 79.3661247464038,
  'arcsecs_pix': 0.820268496350817},
 'wcs_8': {'ra': 206.41889128012915,
  'dec': 79.36609407908034,
  'arcsecs_pix': 0.8202453161017486},
 'wcs_9': {'ra': 206.4164897006368,
  'dec': 79.36614785674082,
  'arcsecs_pix': 0.8201335564467022},
 'wcs_10': {'ra': 206.41695791009187,
  'dec': 79.36603649898518,
  'arcsecs_pix

In [134]:
for i in range(len(wcs_per_file)):
    if 'arcsecs_pix' in wcs_per_file[f'wcs_{i+1}']:
        print(wcs_per_file[f'wcs_{i+1}']['arcsecs_pix'])

0.8201355075772486
0.8201730728024932
0.820167711577099
0.8202298533186253
0.8202477380554389
0.8202309240012386
0.820268496350817
0.8202453161017486
0.8201335564467022
0.8203904906832636
0.8204532911500408
0.8201059183521219
0.8200479156048945
0.8200973829468506
0.8200889317885984


In [135]:
for i in range(len(wcs_per_file)):
    if 'dec' in wcs_per_file[f'wcs_{i+1}']:
        print(wcs_per_file[f'wcs_{i+1}']['dec'])

79.3663646735679
79.36611606931007
79.36618027445387
79.36607385857997
79.36607768884902
79.36596961006465
79.3661247464038
79.36609407908034
79.36614785674082
79.36603649898518
79.36598350552192
79.36617448398289
79.3661834192607
79.36633296464817
79.3663529476514


In [136]:
for i in range(len(wcs_per_file)):
    if 'ra' in wcs_per_file[f'wcs_{i+1}']:
        print(wcs_per_file[f'wcs_{i+1}']['ra'])

206.4241079445839
206.42537579399442
206.42504544177606
206.4227017556861
206.41958727236428
206.41987271355652
206.41885674807222
206.41889128012915
206.4164897006368
206.41695791009187
206.41684704507819
206.41495564737951
206.41321465133998
206.4104399369795
206.41049200883663
